# 70/30 Tape Attack Training Notebook
**Split:** 70% tape-attacked images, 30% clean images in training set  
**Augmentation:** Albumentations pipeline applied to ALL training images to expand effective dataset size  
**Model:** Fine-tuned from `best.pt` (baseline) for 50 epochs

---
### Setup Instructions
1. Upload `archive.zip` (dataset) and `best.pt` (baseline model) to `/content/`
2. Set runtime to **T4 GPU** (Runtime → Change runtime type)
3. Run all cells in order

In [2]:
!pip install ultralytics albumentations --quiet


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import torch
print(f"Is GPU available? {torch.cuda.is_available()}")
# This should print: Is GPU available? True
# If False, change runtime type (top right downwards arrow) to T4 GPU, then restart session

Is GPU available? False


In [6]:
import zipfile
import os

# Must have data.zip (the dataset) in /content (drag it into 'files' on left menu)
# Makes datasets folder and puts unzipped archive into datasets
os.makedirs('/content/datasets', exist_ok=True)
with zipfile.ZipFile('/content/data.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/datasets/data')

print("Dataset extracted.")

OSError: [Errno 30] Read-only file system: '/content'

## Step 1 — Build the 70/30 Robust Training Set

This cell:
- Shuffles all training images
- Applies tape attack to **70%** of them
- Leaves **30%** clean
- Applies **Albumentations augmentation** to ALL images (both clean and attacked)

The augmentation pipeline (flips, brightness/contrast jitter, blur, hue shift, CLAHE) effectively doubles
the training set size and adds visual diversity without needing extra raw data.

In [ ]:
import random
import shutil
import cv2
import os
import numpy as np
from pathlib import Path
import albumentations as A


# ── Attack functions (same as other notebooks) ──────────────────────────────

def apply_digital_tape(image, bbox, color=(50, 50, 50), thickness_ratio=0.2):
    """Applies a horizontal tape strip across the center of the bounding box."""
    h, w, _ = image.shape
    x_center, y_center, b_w, b_h = bbox
    x1 = int((x_center - b_w / 2) * w)
    x2 = int((x_center + b_w / 2) * w)
    tape_h = int((int((y_center + b_h / 2) * h) - int((y_center - b_h / 2) * h)) * thickness_ratio)
    tape_y1 = int(y_center * h - tape_h / 2)
    tape_y2 = int(y_center * h + tape_h / 2)
    cv2.rectangle(image, (x1, tape_y1), (x2, tape_y2), color, -1)
    return image


# ── Albumentations augmentation pipeline ────────────────────────────────────
# Applied to ALL training images (clean + attacked) to boost dataset diversity.
# Each image gets a randomly sampled version of these transforms.

augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    A.CLAHE(clip_limit=2.0, p=0.3),
    A.ImageCompression(quality_lower=75, quality_upper=100, p=0.3),
    A.RandomShadow(p=0.2),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.3))


def parse_label_file(label_path):
    """Returns (class_ids, bboxes) lists from a YOLO label file."""
    class_ids, bboxes = [], []
    if not os.path.exists(label_path):
        return class_ids, bboxes
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                class_ids.append(int(parts[0]))
                bboxes.append([float(x) for x in parts[1:5]])
    return class_ids, bboxes


def create_robust_train_set(base_path, split_ratio=0.7, attack_type="tape"):
    """
    Builds a robust training set:
      - split_ratio (0.7) -> 70% of images get tape attack applied
      - ALL images then receive Albumentations augmentation
      - Saves original AND augmented copy -> ~2x effective dataset size
    """
    train_images_src = os.path.join(base_path, 'train/images')
    train_labels_src = os.path.join(base_path, 'train/labels')

    robust_root = os.path.join(base_path, 'train_robust_70_30')
    robust_images = os.path.join(robust_root, 'images')
    robust_labels = os.path.join(robust_root, 'labels')
    os.makedirs(robust_images, exist_ok=True)
    os.makedirs(robust_labels, exist_ok=True)

    all_images = [f for f in os.listdir(train_images_src) if f.endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(all_images)

    split_idx = int(len(all_images) * split_ratio)
    attack_set = set(all_images[:split_idx])

    print(f"Total training images : {len(all_images)}")
    print(f"  Tape-attacked (70%) : {len(attack_set)}")
    print(f"  Clean         (30%) : {len(all_images) - len(attack_set)}")
    print(f"  + Augmented copies  : {len(all_images)} (one per image)")
    print(f"  Effective total     : ~{len(all_images) * 2}\n")

    skipped = 0
    for img_name in all_images:
        stem = img_name.rsplit('.', 1)[0]
        lbl_name = stem + '.txt'
        src_img_path = os.path.join(train_images_src, img_name)
        src_lbl_path = os.path.join(train_labels_src, lbl_name)

        img = cv2.imread(src_img_path)
        if img is None:
            skipped += 1
            continue

        class_ids, bboxes = parse_label_file(src_lbl_path)

        # ── Apply tape attack to the 70% subset ──
        if img_name in attack_set:
            for bbox in bboxes:
                img = apply_digital_tape(img, bbox)

        # ── Save original (possibly attacked) image ──
        cv2.imwrite(os.path.join(robust_images, img_name), img)
        if os.path.exists(src_lbl_path):
            shutil.copy(src_lbl_path, os.path.join(robust_labels, lbl_name))

        # ── Save augmented copy ──
        if bboxes:
            try:
                result = augment(image=img, bboxes=bboxes, class_labels=class_ids)
                aug_img = result['image']
                aug_bboxes = result['bboxes']
                aug_labels = result['class_labels']
            except Exception:
                aug_img = img
                aug_bboxes = bboxes
                aug_labels = class_ids
        else:
            aug_img = img
            aug_bboxes, aug_labels = [], []

        aug_img_name = stem + '_aug.jpg'
        aug_lbl_name = stem + '_aug.txt'
        cv2.imwrite(os.path.join(robust_images, aug_img_name), aug_img)
        with open(os.path.join(robust_labels, aug_lbl_name), 'w') as f:
            for cls, bbox in zip(aug_labels, aug_bboxes):
                f.write(f"{cls} {' '.join(f'{v:.6f}' for v in bbox)}\n")

    print(f"✅ Robust training set created at: {robust_root}")
    if skipped:
        print(f"⚠️  Skipped {skipped} unreadable images.")


create_robust_train_set('/content/datasets/archive', split_ratio=0.7, attack_type="tape")

## Step 2 — Generate YAMLs

In [ ]:
import yaml

CLASS_NAMES = ['0', '1', '100', '101', '102', '103', '105', '106', '107', '108', '109', '110', '111', '112', '113', '115', '116', '117', '12', '120', '121', '122', '123', '124', '125', '127', '128', '129', '131', '132', '133', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '15', '150', '153', '154', '155', '156', '158', '159', '16', '161', '163', '164', '166', '167', '168', '169', '17', '170', '171', '172', '173', '174', '175', '177', '178', '179', '18', '180', '181', '19', '2', '20', '21', '22', '23', '24', '27', '28', '29', '3', '31', '32', '33', '34', '35', '36', '39', '4', '40', '41', '42', '43', '45', '46', '47', '48', '49', '5', '50', '50 meters between vehicles', '51', '52', '53', '54', '55', '56', '57', '59', '60', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '73', '74', '76', '78', '79', '8', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '93', '94', '95', '97', '98', '99', 'Advance direction sign exit ahead from other road than motorway or expressway', 'Axle weight limit-2ton', 'Bus stop', 'Cattle', 'Crossroad intersection', 'Crossroads', 'Cycle track', 'Cyclist and mopeds rides on carrigeway', 'Cyclists', 'Dangerous shoulder', 'Dip', 'Direction sign exit sign', 'Direction to be followed', 'End of all restrictions', 'End of lane reserved for public transport', 'Falling rocks', 'Filling station', 'First aid post', 'Give way (Yield)', 'Give way -Yield-', 'Guarded level crossing ahead', 'Height limit-3.5m', 'Horn prohibited', 'Hotel', 'Keep left', 'Left curve', 'Level crossing countdown marker', 'Loose gravel', 'Main highways', 'Marking for sharp bends', 'Motorway', 'Narrow bridge', 'Narrow road', 'National border', 'No Left turn', 'No Right turn', 'No entry', 'No entry for Trucks', 'No entry for bicycles', 'No entry for bullock carts', 'No entry for hand carts', 'No entry for motor vehicles', 'No entry for pedestrians', 'No parking', 'No passing', 'No snowmobiles', 'No stopping', 'No vehicles exceeding 12 tonnes', 'No vehicles exceeding length shown', 'No vehicles in both directions', 'No vehicles or combination of vehicles exceeding weight shown', 'Oblique side road junction', 'One-way traffic', 'Parking', 'Parking allowed for 15min', 'Passing without stopping prohibited', 'Pedestrian crossing', 'Priority over oncoming vehicles', 'Refreshments', 'Restaurant', 'Restriction zone', 'Right curve', 'Road hump', 'Road number sign', 'Roadworks', 'Roundabout', 'School', 'Side road junction', 'Slippery road', 'Speed refulcation bump', 'Staggered side road junction', 'Steep ascent', 'Steep descent', 'Steep downhill', 'Steep uphill', 'Stop', 'Stop at customs', 'Straight ahead', 'Symbol plate for specified vehicle or road user category', 'T-junction', 'Telephone', 'Traffic signals', 'Turn Right', 'Turn left', 'Turn left ahead', 'Turn left or straight ahead', 'Turn right ahead', 'Uneven road', 'Unguarded level crossing ahead', 'Unprotected quay', 'Weight limit-5ton', 'Y-junction', 'adjoining way', 'axle weight limit 30tonnes', 'end of the speed limit', 'exit', 'falling rocks (from) left', 'falling rocks -from- left', 'length limit-10m', 'lowspeed zone', 'no entry for bicycles and humans', 'no motorcycles', 'no turning left', 'no u turn', 'snowmobiles', 'speed limit-100', 'speed limit-110', 'speed limit-30', 'speed limit-50', 'speed limit-60', 'speed limit-70', 'speed limit-80', 'speed limit-90', 'speed-limit-120', 'tunnel in 2 km', 'two way traffic', 'warning wild animal']

BASE = '/content/datasets/archive'

# robust.yaml — used during training (points to 70/30 augmented train set)
robust_yaml = {
    'path': BASE,
    'train': 'train_robust_70_30/images',
    'val': 'valid/images',
    'test': 'valid/images',
    'nc': 264,
    'names': CLASS_NAMES
}
with open(f'{BASE}/robust_70_30.yaml', 'w') as f:
    yaml.dump(robust_yaml, f)
print("✅ robust_70_30.yaml written")

# attack.yaml — used during post-training evaluation (val = attacked validation set)
attack_yaml = {
    'path': BASE,
    'train': 'train_robust_70_30/images',
    'val': 'attacked_val/images',
    'test': 'attacked_val/images',
    'nc': 264,
    'names': CLASS_NAMES
}
with open(f'{BASE}/attack_70_30.yaml', 'w') as f:
    yaml.dump(attack_yaml, f)
print("✅ attack_70_30.yaml written")

## Step 3 — Train

In [ ]:
from ultralytics import YOLO

# Fine-tune from the baseline model
model = YOLO('/content/best.pt')

model.train(
    data='/content/datasets/archive/robust_70_30.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    lr0=0.001,
    # augmentation already done in preprocessing, keep YOLO's built-in light
    mosaic=0.5,
    mixup=0.05,
    device=0,
    name='yolov8_robust_70_30'
)

## Step 4 — Build Attacked Validation Set
Tape-attacks all validation images so we can measure robustness.

In [ ]:
def create_attacked_val_set(base_path, attack_type="tape"):
    val_images_src = os.path.join(base_path, 'valid/images')
    val_labels_src = os.path.join(base_path, 'valid/labels')
    attacked_root = os.path.join(base_path, 'attacked_val')
    attacked_images_dest = os.path.join(attacked_root, 'images')
    attacked_labels_dest = os.path.join(attacked_root, 'labels')

    if os.path.exists(attacked_root):
        shutil.rmtree(attacked_root)
    os.makedirs(attacked_images_dest, exist_ok=True)
    os.makedirs(attacked_labels_dest, exist_ok=True)

    all_images = [f for f in os.listdir(val_images_src) if f.endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Attacking {len(all_images)} validation images...")

    for img_name in all_images:
        img = cv2.imread(os.path.join(val_images_src, img_name))
        lbl_name = img_name.rsplit('.', 1)[0] + '.txt'
        src_lbl = os.path.join(val_labels_src, lbl_name)
        dst_lbl = os.path.join(attacked_labels_dest, lbl_name)

        if os.path.exists(src_lbl):
            shutil.copy2(src_lbl, dst_lbl)
            with open(src_lbl, 'r') as f:
                for line in f:
                    parts = line.split()
                    if len(parts) >= 5:
                        bbox = [float(x) for x in parts[1:]]
                        img = apply_digital_tape(img, bbox)

        cv2.imwrite(os.path.join(attacked_images_dest, img_name), img)

    print(f"✅ Attacked validation set created at {attacked_root}")


create_attacked_val_set('/content/datasets/archive', attack_type="tape")

## Step 5 — Evaluate

In [ ]:
robust_model = YOLO('/content/runs/detect/yolov8_robust_70_30/weights/best.pt')

print("--- CLEAN VALIDATION DATA ---")
clean_results = robust_model.val(data='/content/datasets/archive/robust_70_30.yaml')

print("\n--- ATTACKED VALIDATION DATA ---")
attack_results = robust_model.val(data='/content/datasets/archive/attack_70_30.yaml')

print("\n" + "="*60)
print(f"{'Metric':<20} {'Clean Val':>15} {'Attacked Val':>15}")
print("-" * 60)
metrics = ['metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/precision(B)', 'metrics/recall(B)']
labels  = ['mAP50', 'mAP50-95', 'Precision', 'Recall']
for label, key in zip(labels, metrics):
    c = clean_results.results_dict.get(key, float('nan'))
    a = attack_results.results_dict.get(key, float('nan'))
    print(f"{label:<20} {c:>15.4f} {a:>15.4f}")
print("=" * 60)